In [1]:
# =============================================================================
# nb51 - IS THERE A GENUINELY LABEL-FREE VERSION OF THE CLASS-CONDITIONAL
#        SHIFT STATISTIC?
#
# WHY THIS EXISTS. Section 5.10 defines S_cov,c as a domain classifier separating
# source and target instances OF CLASS c. Building the target side requires
# {X_i^t : Y_i^t = c}, that is, the target labels deployment does not have. The
# statistic is therefore an oracle retrospective marker, and any claim that it is
# a deployment-time test is false. That error was withdrawn from the manuscript.
#
# THE QUESTION HERE. Can the same quantity be approximated without labels by
# weighting each target flow by its PREDICTED probability of belonging to c?
#
#     S_soft,c = weighted domain-classifier AUROC, target weights = P_hat(Y=c | x)
#
# If the soft proxy tracks the oracle, the diagnostic can be stated honestly as a
# deployment-time signal. If it does not, that failure is the answer and we say so.
#
# WHAT I EXPECT TO HAPPEN, RECORDED BEFORE RUNNING. The proxy weights by predicted
# class, so flows the shifted model confidently misroutes OUT of c receive almost
# no weight. Misroute rates on the failing classes are already measured and are
# severe: R2L 0.965, scan11 0.563, CIC DoS 0.433, NSL Probe 0.370. The proxy is
# therefore blind to most of the traffic whose drift matters, on precisely the
# classes where the statistic would need to work. I expect it to fail there and to
# succeed on healthy classes, which would make it useless for its intended purpose.
#
# Three variants are tested so the conclusion is not about one arbitrary choice:
#   HARD  : target side = flows PREDICTED c            (argmax assignment)
#   SOFT  : target side weighted by P_hat(Y=c | x)     (soft responsibility)
#   TOPQ  : target side = flows in the top decile of P_hat(Y=c | x)
# =============================================================================
import numpy as np, pandas as pd
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import os, sys, json, shutil, glob, subprocess, time
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
assert PROJECT_ROOT.exists(), 'Drive mount unhealthy; restart runtime and remount'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config
RD=config.REPORTS_DIR; ALPHA=config.ALPHA_PRIMARY
SUB=20000; FOLDS=5; MIN_PER_SIDE=40
print('ready | per-side cap', SUB)


Mounted at /content/drive
ready | per-side cap 20000


In [2]:
# =============================================================================
# Cell 2 - the estimators. The oracle is exactly the nb42 definition so the
# comparison is like for like. The proxies differ ONLY in how the target side is
# selected or weighted; the source side uses source labels, which are available.
# =============================================================================
def scov_weighted(Xs, Xt, wt=None, seed=0, folds=FOLDS):
    """Cross-fitted domain-classifier AUROC. wt gives per-row weights on the TARGET
    side; None means every target row counts equally (the oracle/hard cases)."""
    ns_, nt_ = len(Xs), len(Xt)
    if ns_ < MIN_PER_SIDE or nt_ < MIN_PER_SIDE: return np.nan, ns_, nt_
    m = min(ns_, nt_, SUB)
    rg = np.random.default_rng(seed)
    si = rg.choice(ns_, m, replace=False)
    ti = rg.choice(nt_, m, replace=False)
    X = np.vstack([Xs[si], Xt[ti]])
    y = np.r_[np.zeros(m), np.ones(m)]
    w = np.r_[np.ones(m), (wt[ti] if wt is not None else np.ones(m))]
    k = min(folds, m // 10) if m < 10*folds else folds
    if k < 2: return np.nan, ns_, nt_
    aucs=[]
    for tr, te in StratifiedKFold(k, shuffle=True, random_state=seed).split(X, y):
        mod = HistGradientBoostingClassifier(max_iter=100, random_state=seed).fit(
            X[tr], y[tr], sample_weight=w[tr])
        p = mod.predict_proba(X[te])[:,1]
        # weighted AUROC: weights apply to the target (positive) side
        aucs.append(roc_auc_score(y[te], p, sample_weight=w[te]))
    return float(np.mean(aucs)), ns_, nt_

def variants_for_class(Xs_c, Xt_all, prob_c, y_t, ci, seed):
    """Oracle uses true labels; the three proxies use only predicted probabilities."""
    out={}
    # ORACLE: target side selected by TRUE label (what Section 5.10 does)
    out['oracle'], n_s, _ = scov_weighted(Xs_c, Xt_all[y_t==ci], seed=seed)
    # HARD: target side selected by argmax prediction
    hard = Xt_all[prob_c.argmax_mask]
    out['hard'], _, _ = scov_weighted(Xs_c, hard, seed=seed) if len(hard)>=MIN_PER_SIDE else (np.nan,0,0)
    # SOFT: every target row, weighted by predicted responsibility for c
    out['soft'], _, _ = scov_weighted(Xs_c, Xt_all, wt=prob_c.p, seed=seed)
    # TOPQ: target rows in the top decile of predicted responsibility
    thr = np.quantile(prob_c.p, 0.90)
    topq = Xt_all[prob_c.p >= thr]
    out['topq'], _, _ = scov_weighted(Xs_c, topq, seed=seed) if len(topq)>=MIN_PER_SIDE else (np.nan,0,0)
    out['n_true'] = int((y_t==ci).sum())
    out['n_pred'] = int(prob_c.argmax_mask.sum())
    out['misroute'] = float(1 - (prob_c.argmax_mask & (y_t==ci)).sum()/max((y_t==ci).sum(),1))
    return out

class ProbC:
    def __init__(self, P, ci):
        self.p = P[:, ci]
        self.argmax_mask = P.argmax(1) == ci
print('estimators defined')
print('  oracle = true-label selection (not deployable)')
print('  hard / soft / topq = predicted-probability only (deployable)')


estimators defined
  oracle = true-label selection (not deployable)
  hard / soft / topq = predicted-probability only (deployable)


In [3]:
# =============================================================================
# Cell 3 - assemble features and calibrated probabilities per environment.
# The probabilities are the cached calibrated outputs, so the proxy uses exactly
# what a deployed system would have.
# =============================================================================
DROP={'label','subtype','partition','is_unseen'}
ENVS={}

# ---- NSL-KDD ----
CL=config.CANONICAL_CLASSES; c2i={c:i for i,c in enumerate(CL)}
tr=pd.read_parquet(config.INTERIM_DIR/'nslkdd_train.parquet').reset_index(drop=True)
te=pd.read_parquet(config.INTERIM_DIR/'nslkdd_test.parquet').reset_index(drop=True)
part=pd.read_parquet(config.PROC_DIR/'nslkdd_source_partition_labels.parquet')
tr=tr.assign(partition=part['partition'].values)
F=[c for c in tr.columns if c not in DROP and pd.api.types.is_numeric_dtype(tr[c])]
s=tr[tr.partition=='source_cal_pool']
pf=sorted(glob.glob(str(config.PROC_DIR/'probs_*.npz')))[0]
P_te=np.load(pf)['target'].astype(np.float64)
ENVS['nslkdd']=dict(Xs=s[F].to_numpy(float), ys=s['label'].map(c2i).to_numpy(),
                    Xt=te[F].to_numpy(float), yt=te['label'].map(c2i).to_numpy(),
                    P=P_te, classes=CL)

# ---- UGR'16 ----
UGR=config.DATASETS_DIR/'ugr16'
us=pd.read_parquet(UGR/'july_week5.parquet'); ut=pd.read_parquet(UGR/'august_week1.parquet')
for d in (us,ut): d['label']=d['label'].astype(str).str.strip().str.lower()
UK=['background','dos','scan11','scan44','nerisbotnet']
us=us[us.label.isin(UK)].reset_index(drop=True); ut=ut[ut.label.isin(UK)].reset_index(drop=True)
UCL=sorted(UK); U2I={c:i for i,c in enumerate(UCL)}
FU=[c for c in us.columns if c not in DROP and pd.api.types.is_numeric_dtype(us[c])]
def strat(df,fr,seed,col='label'):
    rg=np.random.default_rng(seed); nm=list(fr); ff=np.array([fr[k] for k in nm],float); big=nm[int(np.argmax(ff))]
    a=pd.Series(index=df.index,dtype=object)
    for _,x in df.groupby(col,sort=True):
        idx=x.index.to_numpy().copy(); rg.shuffle(idx); n=len(idx)
        c=np.floor(ff*n).astype(int); c[nm.index(big)]+=n-c.sum(); k=0
        for a2,q in zip(nm,c): a.loc[idx[k:k+q]]=a2; k+=q
    return a
us=us.assign(partition=strat(us,config.SPLIT_FRACTIONS,20260725).values)
usp=us[us.partition=='source_cal_pool']
uf=sorted((config.DATA_DIR/'ugr16_probs').glob('ugr16__*.npz'))[0]
ENVS['ugr16']=dict(Xs=usp[FU].to_numpy(float), ys=usp['label'].map(U2I).to_numpy(),
                   Xt=ut[FU].to_numpy(float), yt=ut['label'].map(U2I).to_numpy(),
                   P=np.load(uf)['target'].astype(np.float64), classes=UCL)

# ---- CIC-IoT-2023 ----
iot=pd.read_parquet(config.PROC_DIR/'ciciot2023_prepared.parquet')
sp=pd.read_parquet(config.PROC_DIR/'ciciot2023_split.parquet')
iot['side']=sp['side'].values; iot['partition']=sp['partition'].values
FI=json.loads((RD/'ciciot2023_prepared_fingerprint.json').read_text())['features']
ICL=json.loads((RD/'ciciot2023_model_record.json').read_text())['classes_canonical_order']
i2i={c:i for i,c in enumerate(ICL)}
isp=iot[iot.partition=='source_cal_pool']; itg=iot[iot.partition=='target_pool']
inf=sorted((config.DATA_DIR/'ciciot_probs').glob('ciciot2023__*.npz'))[0]
ENVS['ciciot2023']=dict(Xs=isp[FI].to_numpy(float), ys=isp['family'].map(i2i).to_numpy(),
                        Xt=itg[FI].to_numpy(float), yt=itg['family'].map(i2i).to_numpy(),
                        P=np.load(inf)['target'].astype(np.float64), classes=ICL)

for k,v in ENVS.items():
    assert len(v['P'])==len(v['yt']), f"{k}: probability rows {len(v['P'])} != target rows {len(v['yt'])}"
    print(f"  {k:12s} source {len(v['ys']):>7,} target {len(v['yt']):>7,} classes {len(v['classes'])}")
print("\nprobability arrays aligned to target rows: verified")


  nslkdd       source  18,894 target  22,544 classes 5
  ugr16        source  60,000 target 400,000 classes 5
  ciciot2023   source 158,079 target 456,174 classes 8

probability arrays aligned to target rows: verified


In [4]:
# =============================================================================
# Cell 4 - compute the oracle and the three deployable proxies for every class.
# =============================================================================
rows=[]; t0=time.time()
for env,d in ENVS.items():
    Xs,ys,Xt,yt,P,classes = d['Xs'],d['ys'],d['Xt'],d['yt'],d['P'],d['classes']
    for ci,cn in enumerate(classes):
        Xs_c = Xs[ys==ci]
        if len(Xs_c) < MIN_PER_SIDE:
            print(f"  {env:12s} {cn:14s} source too small ({len(Xs_c)}), skipped"); continue
        pc = ProbC(P, ci)
        r = variants_for_class(Xs_c, Xt, pc, yt, ci, seed=abs(hash((env,cn)))%(2**31))
        r.update({'dataset':env,'class':cn})
        rows.append(r)
        o=r['oracle']; s_=r['soft']
        print(f"  {env:12s} {cn:14s} oracle {o:.4f} | hard {r['hard'] if r['hard']==r['hard'] else float('nan'):.4f} "
              f"| soft {s_:.4f} | topq {r['topq'] if r['topq']==r['topq'] else float('nan'):.4f} "
              f"| misroute {r['misroute']:.3f}  [{time.time()-t0:.0f}s]")
V=pd.DataFrame(rows)
print(f"\n{len(V)} classes | {time.time()-t0:.0f}s")


  nslkdd       Normal         oracle 0.7841 | hard 0.8393 | soft 0.8346 | topq 0.8121 | misroute 0.025  [17s]
  nslkdd       DoS            oracle 0.9601 | hard 0.9525 | soft 0.9422 | topq 0.9487 | misroute 0.193  [31s]
  nslkdd       Probe          oracle 0.9792 | hard 0.9752 | soft 0.9600 | topq 0.9779 | misroute 0.306  [39s]
  nslkdd       R2L            oracle 0.9960 | hard 0.9897 | soft 0.9611 | topq 0.9895 | misroute 0.909  [40s]
  nslkdd       U2R            source too small (7), skipped
  ugr16        background     oracle 0.6677 | hard 0.6979 | soft 0.6977 | topq 0.8755 | misroute 0.014  [51s]
  ugr16        dos            oracle 0.5357 | hard 0.5345 | soft 0.5429 | topq 0.5981 | misroute 0.000  [54s]
  ugr16        nerisbotnet    oracle 0.5778 | hard 0.7577 | soft 0.7144 | topq 0.7560 | misroute 0.487  [58s]
  ugr16        scan11         oracle 0.9997 | hard 0.9981 | soft 0.9816 | topq 0.9979 | misroute 0.517  [64s]
  ugr16        scan44         oracle 0.9844 | hard 0.9909 | 

In [5]:
# =============================================================================
# Cell 5 - THE TEST. Does any deployable proxy track the oracle, and does it
# still order coverage failure?
# =============================================================================
cc=pd.read_csv(RD/'class_conditional_scov_vs_coverage.csv')[['dataset','class','coverage','undercoverage']]
M=V.merge(cc, on=['dataset','class'], how='left')
M=M[M.oracle.notna()].copy()
print("ORACLE vs DEPLOYABLE PROXIES")
print(M[['dataset','class','oracle','hard','soft','topq','misroute','undercoverage']]
      .round(4).sort_values('undercoverage',ascending=False).to_string(index=False))

print("\n1. DOES THE PROXY TRACK THE ORACLE?")
for v in ['hard','soft','topq']:
    g=M[M[v].notna()]
    if len(g)<4: continue
    r,p=stats.spearmanr(g[v], g.oracle)
    mae=float((g[v]-g.oracle).abs().mean())
    print(f"   {v:5s} vs oracle: rho {r:+.3f} (p {p:.4f}, n {len(g)}) | mean abs difference {mae:.4f}")

print("\n2. DOES THE PROXY STILL ORDER COVERAGE FAILURE?  (this is what it is FOR)")
r_o,p_o=stats.spearmanr(M.oracle, M.undercoverage)
print(f"   oracle vs undercoverage: rho {r_o:+.3f} (p {p_o:.4f})   <- the target to beat")
for v in ['hard','soft','topq']:
    g=M[M[v].notna()]
    if len(g)<4: continue
    r,p=stats.spearmanr(g[v], g.undercoverage)
    print(f"   {v:5s}  vs undercoverage: rho {r:+.3f} (p {p:.4f}, n {len(g)})")

print("\n3. WHERE DOES THE PROXY BREAK?  (predicted: on high-misroute classes)")
M['gap']=(M.soft-M.oracle).abs()
r_m,p_m=stats.spearmanr(M.misroute, M.gap)
print(f"   misroute rate vs |soft - oracle|: rho {r_m:+.3f} (p {p_m:.4f})")
hi=M[M.misroute>0.35]; lo=M[M.misroute<=0.35]
print(f"   high misroute (>0.35, n={len(hi)}): mean gap {hi.gap.mean():.4f}")
print(f"   low  misroute (n={len(lo)}):        mean gap {lo.gap.mean():.4f}")

print("\nVERDICT")
best=None
for v in ['soft','hard','topq']:
    g=M[M[v].notna()]
    if len(g)<4: continue
    r,_=stats.spearmanr(g[v], g.undercoverage)
    if best is None or r>best[1]: best=(v,r)
if best and best[1] >= 0.6:
    print(f"   The {best[0]} proxy orders coverage failure at rho {best[1]:+.3f} without using")
    print("   target labels. A deployment-time version of the diagnostic is therefore")
    print("   available, and the withdrawn claim can be restated honestly using it.")
elif best:
    print(f"   The best deployable proxy reaches only rho {best[1]:+.3f} against the oracle's")
    print(f"   {r_o:+.3f}. Conditioning on predicted class discards the traffic whose drift")
    print("   matters, so no label-free version of this statistic is available by this route.")
    print("   That is the answer, and it should be reported rather than worked around.")


ORACLE vs DEPLOYABLE PROXIES
   dataset       class  oracle   hard   soft   topq  misroute  undercoverage
    nslkdd         R2L  0.9960 0.9897 0.9611 0.9895    0.9088         0.9202
    nslkdd         DoS  0.9601 0.9525 0.9422 0.9487    0.1928         0.5512
     ugr16      scan11  0.9997 0.9981 0.9816 0.9979    0.5166         0.4150
    nslkdd       Probe  0.9792 0.9752 0.9600 0.9779    0.3057         0.3222
     ugr16      scan44  0.9844 0.9909 0.9572 0.9960    0.3166         0.1510
    nslkdd      Normal  0.7841 0.8393 0.8346 0.8121    0.0246         0.0232
     ugr16 nerisbotnet  0.5778 0.7577 0.7144 0.7560    0.4866         0.0027
     ugr16  background  0.6677 0.6979 0.6977 0.8755    0.0136         0.0008
ciciot2023       Recon  0.5022 0.6109 0.5888 0.6690    0.0924         0.0003
ciciot2023         DoS  0.5025 0.5249 0.5071 0.6652    0.0173         0.0001
ciciot2023        DDoS  0.5006 0.5068 0.5155 0.7820    0.0132        -0.0000
     ugr16         dos  0.5357 0.5345 0.5429 0.

In [ ]:
# =============================================================================
# Cell 6 - save and commit.
# =============================================================================
M.to_csv(RD/'scov_labelfree_proxy.csv', index=False)
res={'question':'is there a deployment-time (label-free) version of the class-conditional '
                'shift statistic S_cov,c?',
     'oracle_definition':'domain classifier on target instances selected by TRUE label',
     'proxies':{'hard':'target selected by argmax prediction',
                'soft':'all target rows weighted by predicted P(Y=c|x)',
                'topq':'target rows in the top decile of predicted P(Y=c|x)'},
     'n_classes':int(len(M)),
     'oracle_vs_undercoverage':float(r_o),
     'proxy_vs_oracle':{v: float(stats.spearmanr(M[M[v].notna()][v], M[M[v].notna()].oracle)[0])
                        for v in ['hard','soft','topq'] if M[v].notna().sum()>3},
     'proxy_vs_undercoverage':{v: float(stats.spearmanr(M[M[v].notna()][v], M[M[v].notna()].undercoverage)[0])
                        for v in ['hard','soft','topq'] if M[v].notna().sum()>3},
     'misroute_vs_gap':float(r_m),
     'cells':M.round(5).to_dict('records')}
(RD/'scov_labelfree_proxy_verdict.json').write_text(json.dumps(res, indent=2, default=str))
print('saved proxy comparison and verdict')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s_,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s_): shutil.copy(s_,dd)
os.chdir(PROJECT_ROOT)
lock=PROJECT_ROOT/'.git'/'index.lock'
if lock.exists() and not subprocess.run(['pgrep','git'],capture_output=True).stdout.strip():
    lock.unlink(); print('removed stale git lock')
for attempt in (1,2):
    git('add','-A',show=False)
    if git('status','--porcelain',show=False).stdout.strip():
        git('commit','-m','nb51: is there a label-free version of the class-conditional shift statistic? oracle vs three deployable proxies')
        r=git('push','-u','origin','main')
        if r.returncode: print('PUSH FAILED. Commit is safe locally.')
        break
    if attempt==1: print('waiting 10s for Drive sync...'); time.sleep(10)
    else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


saved proxy comparison and verdict
